# BRIDGE-JUPYTER-01 — PostgreSQL magics solution

## Goal

Demonstrate one credential-safe, read-only JupySQL workflow against the disposable course PostgreSQL database, including bounded results, named value parameters, pandas conversion, connection cleanup, and an explicit transaction example.

**Level:** Intermediate/advanced  
**Stable lesson ID:** `bridge-jupyter-01`  
**Prerequisites:** Python Day 18, SQL Day 15, Bridge Day 3, and a reset disposable course database.

`%sql` is a line magic for a short statement or result assignment. `%%sql` is a cell magic whose body is multi-line SQL.

## Setup

Set `DS60_DATABASE_URL` in the shell that launches VS Code or Jupyter. The notebook reads it once, never displays it, and rejects any database other than the disposable `advanced_sql_training` target.

In [ ]:
# ruff: noqa: E501 -- JupySQL line magics stay on one physical line.
%load_ext sql

In [ ]:
import os

from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

raw_database_url = os.environ.get("DS60_DATABASE_URL", "").strip()
if not raw_database_url:
    raise RuntimeError(
        "Set DS60_DATABASE_URL in the shell that starts this notebook, then restart the kernel."
    )

course_url = make_url(raw_database_url)
if course_url.get_backend_name() not in {"postgres", "postgresql"}:
    raise RuntimeError("DS60_DATABASE_URL must select PostgreSQL.")
if course_url.database != "advanced_sql_training":
    raise RuntimeError("This lesson is restricted to the disposable course database.")

psycopg_url = course_url.set(drivername="postgresql+psycopg")
engine = create_engine(psycopg_url, pool_pre_ping=True)
assert engine.url.drivername == "postgresql+psycopg"

In [ ]:
%config SqlMagic.displaycon = False
%config SqlMagic.autolimit = 200
%config SqlMagic.displaylimit = 25
%config SqlMagic.autopandas = False
%config SqlMagic.named_parameters = "enabled"

In [ ]:
%sql engine --alias ds60-course

In [ ]:
%sql --connections

## Steps

### 1. Run bounded read-only SQL

The line diagnostic is short. The customer query is easier to review as a cell magic.

In [ ]:
%sql SELECT current_database() AS database_name, current_user AS database_user

In [ ]:
%%sql
SELECT customer_id, full_name, country, segment
FROM training.customers
ORDER BY customer_id
LIMIT 5;

### 2. Bind values with `:name`

The status and amount remain Python data. Neither value is rendered into SQL source.

In [ ]:
order_status = "paid"
minimum_total = 250

In [ ]:
orders_result = %sql SELECT order_id, customer_id, total_amount FROM training.orders WHERE status = :order_status AND total_amount >= :minimum_total ORDER BY total_amount DESC, order_id LIMIT 20

In [ ]:
orders_frame = orders_result.DataFrame()
assert list(orders_frame.columns) == ["order_id", "customer_id", "total_amount"]
assert len(orders_frame) <= 20
orders_frame.head()

### 3. Compare explicit conversion with `autopandas`

`autopandas=True` returns a DataFrame directly. The SQL still has its own `LIMIT` because `displaylimit` affects presentation, not fetched row count, and pandas display rules apply in this mode.

In [ ]:
%config SqlMagic.autopandas = True
customer_frame = %sql SELECT customer_id, full_name, country FROM training.customers ORDER BY customer_id LIMIT 10
%config SqlMagic.autopandas = False

In [ ]:
assert customer_frame.shape[0] <= 10
assert {"customer_id", "full_name", "country"}.issubset(customer_frame.columns)

### 4. Keep code generation out of the value path

Jinja `{{value}}` renders SQL code before execution and therefore requires trusted, reviewed input. Named `:value` syntax uses the parameter boundary and is the correct choice for data values. Neither form safely binds a table or column name. Keep notebook identifiers static; application code can combine an allowlist with `psycopg.sql.Identifier` when dynamism is truly required.

### 5. Make transaction ownership visible

JupySQL autocommit defaults to true. This lesson performs only reads. The SQLAlchemy block below demonstrates an explicit transaction context without changing data.

In [ ]:
with engine.begin() as connection:
    observed_database = connection.scalar(text("SELECT current_database()"))

assert observed_database == "advanced_sql_training"

### 6. Move reusable effects into Psycopg code

Magics are excellent for bounded exploration. Reusable typed queries, multiple writes, COPY, streaming, retry classification, pooling, async work, cancellation, structured logging, and unit-test seams belong in application modules using Psycopg or a deliberately chosen SQLAlchemy layer.

## Checks

The completed aggregate binds country and threshold as data, preserves static identifiers, orders deterministically, and caps output.

In [ ]:
exercise_country = "US"
exercise_minimum_total = 500

In [ ]:
customer_totals = %sql SELECT c.customer_id, c.full_name, COALESCE(sum(o.total_amount), 0) AS lifetime_total FROM training.customers AS c LEFT JOIN training.orders AS o USING (customer_id) WHERE c.country = :exercise_country GROUP BY c.customer_id, c.full_name HAVING COALESCE(sum(o.total_amount), 0) >= :exercise_minimum_total ORDER BY lifetime_total DESC, c.customer_id LIMIT 10

In [ ]:
exercise_frame = customer_totals.DataFrame()
assert list(exercise_frame.columns) == ["customer_id", "full_name", "lifetime_total"]
assert len(exercise_frame) <= 10
exercise_frame

The safety check is behavioral: no connection literal or saved output exists; the result is bounded; the two inputs are named parameters; identifiers are fixed course objects; the query is read-only; and the connection is closed explicitly.

In [ ]:
%sql --close ds60-course

In [ ]:
engine.dispose()

## Next Steps

Return to Bridge Days 3–5 to package exploratory SQL behind typed Psycopg functions and fake-backed tests. Continue to BRIDGE-OPS-01 to connect migration checks, request IDs, retry policy, health/readiness, metrics, and forward-fix versus rollback evidence.